# **ANÁLISIS FILOGENÉTICO PARA LA DETERMINACIÓN DE IDENTIDAD DE HA Y NA**

## **Preparación del archivo de secuencias de referencias** (Solo en caso de no contar con el archivo de referencia prearmado, o para modificar parámetros)

Se prepararon archivos fasta con la secuencia de aminoácidos los segmentos HA y NA, por separado. 
Para ellos desde NCBI virus se filtran los datos aplicando los siguientes criterios:

Para HA
- Identifier and Classification: Virus/Taxonomy: **2955291**
- Sequence Quality: Nucleotide Completeness: **complete**
- Location and Source: Host: **Non-human**
- Genome Organization: Segment: **4**
- 
Para NA
- Identifier and Classification: Virus/Taxonomy: **2955291**
- Sequence Quality: Nucleotide Completeness: **complete**
- Location and Source: Host: **Non-human**
- Genome Organization: Segment: **6**

El archivo se descarga con formato FASTA de proteins, aplicando los siguientes campos, en orden estricto:
- Accesion
- Genotype
- Host

Renombrar los archivos como:
- NCBIvirus_HA.faa
- NCBIvirus_NA.faa

Limpiar headers de acuerdo al fasta con:

In [1]:
# Para HA
awk -F'|' 'BEGIN{OFS="|"} 
  /^>/ {
    gsub(/ /, "", $1)
    sub(/N[0-9]+$/, "", $2)
    gsub(/ /, "_", $3)
    gsub(/ /, "_", $4)
    print $1,$2,$3,$4
    next
  }
  { print } 
' NCBIvirus_HA.faa >NCBIvirus_HA_clean_header.faa

# Para NA
awk -F'|' 'BEGIN{OFS="|"} 
  /^>/ {
    gsub(/ /, "", $1)
    sub(/^H[0-9]+/, "", $2)
    gsub(/ /, "_", $3)
    gsub(/ /, "_", $4)
    print $1,$2,$3,$4
    next
  }
  { print } 
' NCBIvirus_NA.faa >NCBIvirus_NA_clean_header.faa

awk: línea ord.:1: fatal: no se puede abrir el fichero «NCBIvirus_HA.faa» para lectura: No existe el fichero o el directorio
awk: línea ord.:1: fatal: no se puede abrir el fichero «NCBIvirus_NA.faa» para lectura: No existe el fichero o el directorio


: 2

Reducir redundancias con:

In [ ]:
conda activate filogenia
cd-hit -i NCBIvirus_HA_clean_header.faa -o NCBIvirus_HA_cdhit.faa -c 0.95
cd-hit -i NCBIvirus_NA_clean_header.faa -o NCBIvirus_NA_cdhit.faa -c 0.95

OPCIONAL: Revisar header con: <br>
grep ">" ./SECS/NCBIvirus_HA_cdhit.faa | sort -t'|' -k2,2
Los scripts de graficación están diseñados para considerar 13 subtipos de HA (1-13) y 9 subtipos de NA (1-9)

En caso de querer eliminar secuencias usar seqkit, por ejemplo: <br>
seqkit grep -v -p "AUI42271.1|mixed|Gallus_gallus|Bangladesh" NCBIvirus_NA_cdhit.faa >NCBIvirus_NA_cdhit_clean.faa

Los archivos resultantes se renombran a: <br>
**NCBIvirus_HA_ref.faa**<br>
**nCBIvirus_HA_ref.faa**

y se depositan en:<br>
**/backup/DATABASES/UASIP/NCBIVIRUS_REF**

Dichos archivos se copiarán en cada análisis de muestras problemas.


## **ANALISIS FILOGENÉTICO**

Especificar nombre de muestra:

In [ ]:
MUESTRA=""

En la carpeta de la muestra problema, crear las carpetas FILOGENIA/TREE e ingresar en FILOGENIA:

In [ ]:
mkdir -p FILOGENIA/TREE && cd FILOGENIA

Copiar archivos de referencia

In [ ]:
cp /backup/DATABASES/UASIP/NCBIVIRUS_REF/NCBIvirus_* .

Añadir manualmente la secuencia de la proteína que codifica a HA y NA de la muestra problema en el respectivo archivo de referencia. <br>
Incluir sólamente la secuencia desde la metionina inicial hasta término. <br>
El header debe incluir el código de la muestra.

Realizar el alineamiento múltiple de secuencias. No se realizará ningún recorte posterior.

In [ ]:
conda activate msa
muscle -align NCBIvirus_HA_ref.faa -output ${MUESTRA}_NCBIvirus_HA.aln
muscle -align NCBIvirus_NA_ref.faa -output ${MUESTRA}_NCBIvirus_NA.aln

In [ ]:
conda activate filogenia
# Si no se sabe el modelo y número de nucleos, o se usan secuencias nuevas usar, por ejemplo para HA:
#iqtree -s ${MUESTRA}_NCBIvirus_HA.aln -m MFP -B 1000 -pre TREE/NA_tree -T AUTO

#PARA HA:
iqtree -s ${MUESTRA}_NCBIvirus_HA.aln -m FLU+R5 -B 1000 -pre TREE/${MUESTRA}_HA -T 7

#PARA NA:
iqtree -s ${MUESTRA}_NNCBIvirus_NA.aln -m FLU+R5 -B 1000 -pre TREE/${MUESTRA}_NA -T 8

Correr los siguientes scripts de R para dibujar los árboles.

In [ ]:
conda activate R
./graficar_filogenia.R  --input_HA ./TREE/${MUESTRA}_HA.contree --input_NA ./TREE/${MUESTRA}_NA.contree --aln_HA ${MUESTRAN}_NCBIvirus_HA.aln --aln_NA ${MUESTRAN}_NCBIvirus_NA.aln --muestra $MUESTRA